# GhostID v4: Open-Set Evaluation, Baselines, Ablations

Run top to bottom on Kaggle (GPU on, dataset attached). Everything printed here is what you may put on the resume and README.

| Section | Output you use |
|---|---|
| 3-4 | Open-set EER/AUC on **unseen users**, GhostID vs baselines, mean ± std over seeds |
| 5 | Ablations: ArcFace vs softmax, with vs without ratio features, enrollment size |
| 6 | Poisoning resistance over all victim/attacker pairs |
| 7 | Corrected v3 numbers (FAR/FRR/EER at product thresholds), ONNX export, size, latency |

Set `QUICK = True` first for a 1-seed smoke test, then set it back to `False` for the real run.

In [ ]:
%pip install -q onnx onnxruntime

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
warnings.filterwarnings('ignore')

DATA_PATH   = '/kaggle/input/datasets/yogeshrayal/dataset/DSL-StrongPasswordData.csv'
OUT_DIR     = '.'
SEEDS       = [0, 1, 2, 3, 4]
N_HOLDOUT   = 10               # users never seen in training (open-set)
EPOCHS      = 60
BATCH_SIZE  = 64
ENROLL_K    = [2, 5, 10, 50]   # enrollment samples per user
THRESHOLDS  = {'HARD_STOP': 40, 'TYPING_CHALLENGE': 69, 'SOFT_NUDGE': 84}
QUICK       = False            # True = 1 seed, 20 epochs (smoke test)
if QUICK:
    SEEDS, EPOCHS = [0], 20
print('seeds', SEEDS, 'epochs', EPOCHS)

In [ ]:
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
raw_cols = [c for c in df.columns if c not in ['subject', 'sessionIndex', 'rep']]
H_cols = [c for c in raw_cols if c.startswith('H.')]
for i in range(len(H_cols) - 1):
    df[f'ratio_{i}'] = df[H_cols[i]] / (df[H_cols[i + 1]] + 1e-8)
ratio_cols = [c for c in df.columns if c.startswith('ratio_')]
FEATS = {'with_ratio': raw_cols + ratio_cols, 'raw_only': raw_cols}
subjects = df['subject'].values
users = np.array(sorted(df['subject'].unique()))
print('rows', len(df), 'users', len(users),
      'features: with_ratio', len(FEATS['with_ratio']), 'raw_only', len(FEATS['raw_only']))
# Rows are in file order = chronological per user (sessionIndex, rep).

## 1. Metrics and protocol helpers
Protocol: for each held-out user, enroll with the first `k` samples, genuine probes = the user's remaining samples, impostor probes = **all** samples of the other held-out users. Primary metric = mean per-user EER (the standard CMU benchmark convention).

In [ ]:
def per_user_eer(gen, imp):
    y = np.r_[np.ones(len(gen)), np.zeros(len(imp))]
    s = np.r_[gen, imp]
    fpr, tpr, _ = roc_curve(y, s)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[i] + fnr[i]) / 2, auc(fpr, tpr)

def pooled_eer(G, I):
    return per_user_eer(G, I)

def eval_users(Z, rows_by_user, k, score_fn):
    # Z: feature or embedding matrix aligned with df rows.
    # rows_by_user: {user: row indices in chronological order}
    ulist = list(rows_by_user.keys())
    eers, aucs, G, I = [], [], [], []
    for u in ulist:
        r = rows_by_user[u]
        enroll, gen = Z[r[:k]], Z[r[k:]]
        imp = np.concatenate([Z[rows_by_user[v]] for v in ulist if v != u])
        g, i_ = score_fn(enroll, gen), score_fn(enroll, imp)
        e, a = per_user_eer(g, i_)
        eers.append(e); aucs.append(a); G.append(g); I.append(i_)
    return dict(eer=float(np.mean(eers)), eer_std_users=float(np.std(eers)),
                auc=float(np.mean(aucs)), G=np.concatenate(G), I=np.concatenate(I),
                per_user=eers)

def sc_cosine(E, P):                      # GhostID: cosine to mean enrolled embedding, x100
    b = E.mean(0); b = b / np.linalg.norm(b)
    return (P @ b) * 100

def sc_euclid(E, P):
    return -np.linalg.norm(P - E.mean(0), axis=1)

def make_scaled_manhattan(global_mad):
    def f(E, P):
        m = E.mean(0)
        mad = np.maximum(np.abs(E - m).mean(0), 0.5 * global_mad)   # floor: tiny k gives mad ~ 0
        return -(np.abs(P - m) / mad).sum(1)
    return f

def sc_mahalanobis(E, P):
    lw = LedoitWolf().fit(E)
    d = P - E.mean(0)
    return -np.einsum('ij,jk,ik->i', d, lw.precision_, d)

def rows_for(user_list, restrict=None):
    out = {}
    for u in user_list:
        r = np.where(subjects == u)[0]
        if restrict is not None:
            r = r[np.isin(r, restrict)]
        out[u] = r
    return out

## 2. Model (same architecture as v3, plus a softmax head for the ablation)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

class GhostIDEncoder(nn.Module):
    def __init__(self, input_size=41, hidden_size=256, num_layers=2, embed_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.fc = nn.Linear(hidden_size, embed_dim)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(self.layer_norm(out[:, -1, :]))
        return nn.functional.normalize(self.fc(out), p=2, dim=1)

class ArcFaceLoss(nn.Module):
    def __init__(self, embed_dim=128, num_classes=51, s=64.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embed_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m, self.sin_m = np.cos(m), np.sin(m)
        self.th = np.cos(np.pi - m)
        self.mm = np.sin(np.pi - m) * m
    def forward(self, emb, labels):
        cosine = nn.functional.linear(emb, nn.functional.normalize(self.weight))
        cosine = cosine.clamp(-1 + 1e-7, 1 - 1e-7)
        sine = torch.sqrt(1.0 - cosine ** 2)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        out = (one_hot * phi + (1.0 - one_hot) * cosine) * self.s
        return nn.CrossEntropyLoss()(out, labels)

class SoftmaxHead(nn.Module):               # ablation: plain softmax on normalized embeddings
    def __init__(self, embed_dim, num_classes, scale=16.0):
        super().__init__()
        self.fc, self.scale = nn.Linear(embed_dim, num_classes), scale
    def forward(self, emb, labels):
        return nn.functional.cross_entropy(self.fc(emb * self.scale), labels)

def train_encoder(X_tr, y_tr, loss_type, epochs, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    d = X_tr.shape[1]
    enc = GhostIDEncoder(input_size=d).to(device)
    ncls = int(y_tr.max()) + 1
    head = (ArcFaceLoss(128, ncls) if loss_type == 'arcface' else SoftmaxHead(128, ncls)).to(device)
    opt = torch.optim.AdamW(list(enc.parameters()) + list(head.parameters()), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xt = torch.FloatTensor(X_tr.reshape(-1, 1, d)).to(device)
    yt = torch.LongTensor(y_tr).to(device)
    n = len(Xt)
    for ep in range(epochs):
        enc.train(); head.train()
        perm = torch.randperm(n, device=device)
        for b in range(0, n, BATCH_SIZE):
            ib = perm[b:b + BATCH_SIZE]
            opt.zero_grad()
            loss = head(enc(Xt[ib]), yt[ib])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
            opt.step()
        sched.step()
    enc.eval()
    return enc

@torch.no_grad()
def embed(enc, X):
    enc.eval(); out = []
    d = X.shape[1]
    for b in range(0, len(X), 4096):
        xb = torch.FloatTensor(X[b:b + 4096].reshape(-1, 1, d)).to(device)
        out.append(enc(xb).cpu().numpy())
    return np.concatenate(out)

## 3-5. Main experiment: open-set EER, baselines, ablations
Per seed: pick 10 users at random as **held out** (never seen in training), train on the other 41, evaluate on the held-out 10. The scaler is fit on training users only (v3 fit it on everyone, which leaks).

In [ ]:
rows = []
seed0 = {}
t0 = time.time()

SPECS = [('GhostID (ArcFace, 41 feats)', 'with_ratio', 'arcface'),
         ('Ablation: softmax loss',       'with_ratio', 'softmax'),
         ('Ablation: no ratio feats',     'raw_only',   'arcface')]

for seed in SEEDS:
    rng = np.random.RandomState(seed)
    perm = rng.permutation(users)
    holdout, train_users = perm[:N_HOLDOUT], perm[N_HOLDOUT:]
    tr_mask = np.isin(subjects, train_users)
    umap = {u: i for i, u in enumerate(train_users)}
    y_tr = np.array([umap[s] for s in subjects[tr_mask]])
    ho_rows = rows_for(holdout)

    # --- baselines (raw 31 timing features, scaler from training users) ---
    sc = StandardScaler().fit(df.loc[tr_mask, FEATS['raw_only']].values)
    X31 = sc.transform(df[FEATS['raw_only']].values).astype(np.float32)
    gmad = np.mean([np.abs(X31[subjects == u] - X31[subjects == u].mean(0)).mean(0) for u in train_users], axis=0)
    base_fns = [('Baseline: Euclidean', sc_euclid, 1),
                ('Baseline: Scaled Manhattan', make_scaled_manhattan(gmad), 1),
                ('Baseline: Mahalanobis (shrinkage)', sc_mahalanobis, 10)]
    for name, fn, kmin in base_fns:
        for k in ENROLL_K:
            if k < kmin: continue
            r = eval_users(X31, ho_rows, k, fn)
            rows.append(dict(seed=seed, method=name, k=k, eer=r['eer'], auc=r['auc']))

    # --- GhostID + ablations ---
    for name, fs, loss_type in SPECS:
        sc = StandardScaler().fit(df.loc[tr_mask, FEATS[fs]].values)
        Xall = sc.transform(df[FEATS[fs]].values).astype(np.float32)
        enc = train_encoder(Xall[tr_mask], y_tr, loss_type, EPOCHS, seed)
        Z = embed(enc, Xall)
        for k in ENROLL_K:
            r = eval_users(Z, ho_rows, k, sc_cosine)
            rows.append(dict(seed=seed, method=name, k=k, eer=r['eer'], auc=r['auc']))
            if seed == SEEDS[0] and name.startswith('GhostID') and k == 2:
                seed0 = dict(Z=Z, holdout=holdout, G=r['G'], I=r['I'])
        print(f'seed {seed} | {name} done | {time.time()-t0:.0f}s')

res = pd.DataFrame(rows)
res.to_csv(f'{OUT_DIR}/open_set_results_raw.csv', index=False)

In [ ]:
summ = (res.groupby(['method', 'k'])
          .agg(eer_mean=('eer', 'mean'), eer_std=('eer', 'std'), auc_mean=('auc', 'mean'), auc_std=('auc', 'std'))
          .reset_index())
summ['EER %'] = (summ.eer_mean * 100).round(2).astype(str) + ' ± ' + (summ.eer_std.fillna(0) * 100).round(2).astype(str)
summ['AUC']   = summ.auc_mean.round(3).astype(str) + ' ± ' + summ.auc_std.fillna(0).round(3).astype(str)
table = summ[['method', 'k', 'EER %', 'AUC']].sort_values(['k', 'method'])
table.to_csv(f'{OUT_DIR}/open_set_summary.csv', index=False)
print('Open-set results on', N_HOLDOUT, 'held-out users, mean ± std over', len(SEEDS), 'seeds (k = enrollment samples)')
print(table.to_string(index=False))

In [ ]:
# Figure for README/report: score distribution + ROC, open-set, k=2, seed 0, pooled over held-out users
G, I = seed0['G'], seed0['I']
fpr, tpr, _ = roc_curve(np.r_[np.ones(len(G)), np.zeros(len(I))], np.r_[G, I])
eer_p, auc_p = pooled_eer(G, I)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].hist(G, bins=50, alpha=.7, label=f'Genuine (n={len(G)})')
ax[0].hist(I, bins=50, alpha=.7, label=f'Impostor (n={len(I)})')
for name, t in THRESHOLDS.items(): ax[0].axvline(t, ls='--', alpha=.5)
ax[0].set_title('Open-set: unseen users, k=2'); ax[0].set_xlabel('Score'); ax[0].legend()
ax[1].plot(fpr, tpr, label=f'AUC={auc_p:.3f}, pooled EER={eer_p*100:.2f}%'); ax[1].plot([0, 1], [0, 1], 'k--', alpha=.3)
ax[1].set_xlabel('FAR'); ax[1].set_ylabel('TAR'); ax[1].set_title('ROC'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/open_set_analysis.png', dpi=150); plt.show()
print(f'pooled (seed 0, k=2): EER {eer_p*100:.2f}%  AUC {auc_p:.3f}')

## 6. Poisoning resistance, quantified
Every victim/attacker pair among the held-out users: 50 genuine probes, then an impostor takes over for 100 probes. The baseline adapts (EMA) when score >= SILENT_PASS, as in the product. Compared against a frozen baseline.

In [ ]:
def poisoning_sim(Z, holdout, k=2, n_gen=50, n_imp=100, silent=85, hard=40, alpha=0.92, adaptive=True):
    out = []
    for u in holdout:
        iu = np.where(subjects == u)[0]
        for v in holdout:
            if u == v: continue
            iv = np.where(subjects == v)[0]
            b0 = Z[iu[:k]].mean(0); b0 = b0 / np.linalg.norm(b0); b = b0.copy()
            seq = [Z[i] for i in iu[k:k + n_gen]] + [Z[i] for i in iv[k:k + n_imp]]
            sc = []
            for z in seq:
                s = float(z @ b) * 100; sc.append(s)
                if adaptive and s >= silent:
                    b = alpha * b + (1 - alpha) * z; b = b / np.linalg.norm(b)
            sc = np.array(sc); g, im = sc[:n_gen], sc[n_gen:]
            out.append(dict(genuine_pass=(g >= silent).mean(),
                            imp_leak=(im >= silent).mean(),
                            imp_hard_stop=(im < hard).mean(),
                            first_hard_stop=int(np.argmax(im < hard)) if (im < hard).any() else n_imp,
                            drift=float(b0 @ b)))
    return pd.DataFrame(out)

pz = {}
for mode in (True, False):
    d = poisoning_sim(seed0['Z'], seed0['holdout'], adaptive=mode)
    pz['adaptive baseline' if mode else 'frozen baseline'] = {
        'pairs': len(d),
        'genuine probes >= 85 (%)':        round(d.genuine_pass.mean() * 100, 1),
        'impostor probes >= 85 (leak, %)': round(d.imp_leak.mean() * 100, 1),
        'impostor probes < 40 (hard stop, %)': round(d.imp_hard_stop.mean() * 100, 1),
        'median probes to first hard stop': float(d.first_hard_stop.median()),
        'mean baseline drift (cosine)':    round(d.drift.mean(), 3)}
pd.DataFrame(pz).to_csv(f'{OUT_DIR}/poisoning_summary.csv')
print(pd.DataFrame(pz).to_string())

## 7. Corrected v3 numbers (closed-set, the product setting) + ONNX export
Same 80/20 split as v3, but: scaler fit on train only, impostors = **all** other users' test samples (not 20 random), per-user and pooled EER, FAR/FRR at your product thresholds. This is the closed-set number: label it "enrolled users" everywhere, never as the headline.

In [ ]:
tr_idx, te_idx = [], []
for u in users:
    ii = np.where(subjects == u)[0]; s = int(len(ii) * 0.8)
    tr_idx += list(ii[:s]); te_idx += list(ii[s:])
tr_idx, te_idx = np.array(tr_idx), np.array(te_idx)

fs = 'with_ratio'
scaler = StandardScaler().fit(df.loc[tr_idx, FEATS[fs]].values)
Xall = scaler.transform(df[FEATS[fs]].values).astype(np.float32)
le = {u: i for i, u in enumerate(users)}
y_all = np.array([le[s] for s in subjects])

final_enc = train_encoder(Xall[tr_idx], y_all[tr_idx], 'arcface', EPOCHS, 0)
Zc = embed(final_enc, Xall)

r = eval_users(Zc, rows_for(users, restrict=te_idx), 2, sc_cosine)
G, I = r['G'], r['I']
eer_p, auc_p = pooled_eer(G, I)
print(f'CLOSED-SET (enrolled users, {len(users)} users)')
print(f'  per-user EER {r["eer"]*100:.2f}% (std across users {r["eer_std_users"]*100:.2f})  AUC {r["auc"]:.4f}')
print(f'  pooled   EER {eer_p*100:.2f}%  AUC {auc_p:.4f}   genuine n={len(G)}  impostor n={len(I)}')
for name, t in THRESHOLDS.items():
    print(f'  {name:17s} (>= {t})  FAR {np.mean(I >= t)*100:.2f}%  FRR {np.mean(G < t)*100:.2f}%')

In [ ]:
import onnx, onnxruntime as ort
final_enc.eval().cpu()
torch.onnx.export(final_enc, torch.randn(1, 1, 41), f'{OUT_DIR}/ghostid_encoder.onnx',
                  export_params=True, opset_version=14, do_constant_folding=True,
                  input_names=['features'], output_names=['embedding'],
                  dynamic_axes={'features': {0: 'batch_size'}, 'embedding': {0: 'batch_size'}})
onnx.checker.check_model(onnx.load(f'{OUT_DIR}/ghostid_encoder.onnx'))
sess = ort.InferenceSession(f'{OUT_DIR}/ghostid_encoder.onnx')
x = np.random.randn(1, 1, 41).astype(np.float32)
for _ in range(50): sess.run(None, {'features': x})
t = time.perf_counter()
for _ in range(2000): o = sess.run(None, {'features': x})
lat = (time.perf_counter() - t) / 2000 * 1000
size_kb = os.path.getsize(f'{OUT_DIR}/ghostid_encoder.onnx') / 1024
print(f'ONNX size {size_kb:.1f} KB | latency {lat:.3f} ms/query (CPU) | embedding norm {np.linalg.norm(o[0]):.4f}')
json.dump({'mean': scaler.mean_.tolist(), 'scale': scaler.scale_.tolist()},
          open(f'{OUT_DIR}/scaler_params.json', 'w'), indent=2)
print('saved ghostid_encoder.onnx + scaler_params.json (use these two together)')

## What to copy back into the repo
1. `open_set_summary.csv`, `poisoning_summary.csv`, `open_set_analysis.png` -> `ml/analysis/`
2. `ghostid_encoder.onnx`, `scaler_params.json` -> `backend/ml/` (they are a matched pair)
3. Save this notebook **with outputs** (Kaggle: Save Version -> Save & Run All) and commit it to `ml/notebooks/`
4. Replace the README benchmark table with the printed numbers, state the protocol, and add a Limitations section